In [1]:
import sys
print(sys.executable)

/opt/homebrew/Caskroom/miniconda/base/envs/review-nlp/bin/python


In [2]:
from sqlalchemy import create_engine, text
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

In [3]:
# 1. connect_to_db() v
# 2. get_reviews_for_business(business_id) v
# 3. create_analysis_run() v
# 4. generate_embeddings(reviews) v
# 5. cluster_embeddings() v
# 6. label_topics() v
# 7. run_sentiment()
# 8. write_topics_to_db()
# 9. write_review_analysis_to_db()
# 10. mark_run_complete()

# Connecting to database

In [4]:
DB_URL = 'postgresql+psycopg2://macbookpro@localhost:5432/review_intelligence'
engine = create_engine(DB_URL)

In [5]:
result = engine.connect().execute(text('SELECT current_user, current_database();')).fetchone()
result

('macbookpro', 'review_intelligence')

### Retrieving sentiment text from database

In [6]:
query = """
SELECT *
FROM review
"""

df_reviews = pd.read_sql(query, engine)
df_reviews.head()

,review_id,business_id,source,rating,text,review_date,created_at
0,e91be263-ec1d-4e92-b399-877546206b10,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,5,Makanannya enak banget! Bakal balik lagi.,2026-02-10,2026-02-16 22:49:52.637856
1,0bed4fca-68d6-4d27-9ef5-68f4ef90f3fe,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,2,Pengiriman lama dan makanan sudah dingin.,2026-02-11,2026-02-16 22:49:52.637856
2,92d8d828-6bd3-4700-9a2f-1156e297e468,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,3,Rasanya ok tapi porsinya kecil.,2026-02-11,2026-02-16 22:49:52.637856
3,d8376516-57a3-4b85-9f5b-87c6b1d9b49d,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,1,"Packaging bocor, kuah tumpah semua.",2026-02-12,2026-02-16 22:49:52.637856
4,bb14fe7c-617f-4ccb-beb2-f0ff7fa8a1f2,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,4,"Harga sesuai, pelayanan ramah.",2026-02-12,2026-02-16 22:49:52.637856


In [7]:
len(df_reviews)

40

### light cleaning the text

In [8]:
import re

def clean_text(text):
    text = text.strip() # remove white spaces before and after the text
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'http\S+', '', text)  # remove URLs
    return text

In [9]:
df = df_reviews.copy()
df['text_clean'] = df.text.apply(clean_text)

In [10]:
df

,review_id,business_id,source,rating,text,review_date,created_at,text_clean
0,e91be263-ec1d-4e92-b399-877546206b10,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,5,Makanannya enak banget! Bakal balik lagi.,2026-02-10,2026-02-16 22:49:52.637856,Makanannya enak banget! Bakal balik lagi.
1,0bed4fca-68d6-4d27-9ef5-68f4ef90f3fe,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,2,Pengiriman lama dan makanan sudah dingin.,2026-02-11,2026-02-16 22:49:52.637856,Pengiriman lama dan makanan sudah dingin.
2,92d8d828-6bd3-4700-9a2f-1156e297e468,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,3,Rasanya ok tapi porsinya kecil.,2026-02-11,2026-02-16 22:49:52.637856,Rasanya ok tapi porsinya kecil.
3,d8376516-57a3-4b85-9f5b-87c6b1d9b49d,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,1,"Packaging bocor, kuah tumpah semua.",2026-02-12,2026-02-16 22:49:52.637856,"Packaging bocor, kuah tumpah semua."
4,bb14fe7c-617f-4ccb-beb2-f0ff7fa8a1f2,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,4,"Harga sesuai, pelayanan ramah.",2026-02-12,2026-02-16 22:49:52.637856,"Harga sesuai, pelayanan ramah."
5,3c35c1d3-b175-49e5-814b-bd1d49eae844,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,2,CS susah dihubungi. Balasan lama.,2026-02-13,2026-02-16 22:49:52.637856,CS susah dihubungi. Balasan lama.
6,31cb85e2-59ad-4093-99a2-b78b9dce480e,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,5,Cepat banget dan rasanya konsisten.,2026-02-13,2026-02-16 22:49:52.637856,Cepat banget dan rasanya konsisten.
7,0d885c41-666b-4624-934e-13f8f9f7f3b3,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,3,Tempatnya agak sempit tapi bersih.,2026-02-14,2026-02-16 22:49:52.637856,Tempatnya agak sempit tapi bersih.
8,07b4ac90-481a-4cd6-b865-adb683891406,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,4,"Promo menarik, jadi worth it.",2026-02-14,2026-02-16 22:49:52.637856,"Promo menarik, jadi worth it."
9,5c51f552-ef87-4266-867a-3196377602ad,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,2,Terlalu asin buat saya.,2026-02-15,2026-02-16 22:49:52.637856,Terlalu asin buat saya.


### Embedding Reviews

In [11]:
model = SentenceTransformer('intfloat/multilingual-e5-small')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
df["input_text"] = df["text_clean"].apply(
    lambda x: "passage: " + x
)


embeddings = model.encode(
    df['input_text'].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

In [13]:
embeddings.shape

(40, 384)

In [14]:
embeddings

array([[ 0.0370748 ,  0.00876136, -0.00566634, ...,  0.05045824,
         0.06013994,  0.05515188],
       [ 0.03657049,  0.02349837, -0.03030078, ...,  0.07048056,
         0.04803155,  0.04247599],
       [ 0.03459861,  0.0078005 , -0.04466819, ...,  0.05384674,
         0.04100662,  0.05925691],
       ...,
       [ 0.01690088, -0.03104838, -0.05896795, ...,  0.06005201,
         0.06476963,  0.06069675],
       [ 0.02865068,  0.01287158, -0.03278923, ...,  0.02319004,
         0.04676758,  0.02972757],
       [ 0.07431573, -0.00065036, -0.05101628, ...,  0.02291125,
         0.04601803,  0.06187063]], shape=(40, 384), dtype=float32)

### Clustering Reviews

In [15]:
from sklearn.cluster import KMeans

In [16]:
k = 6

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(embeddings)
df["cluster"] = clusters

In [17]:
df["cluster"].value_counts()

cluster
2    10
3    10
0     8
1     7
4     3
5     2
Name: count, dtype: int64

In [18]:
df

,review_id,business_id,source,rating,text,review_date,created_at,text_clean,input_text,cluster
0,e91be263-ec1d-4e92-b399-877546206b10,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,5,Makanannya enak banget! Bakal balik lagi.,2026-02-10,2026-02-16 22:49:52.637856,Makanannya enak banget! Bakal balik lagi.,passage: Makanannya enak banget! Bakal balik l...,2
1,0bed4fca-68d6-4d27-9ef5-68f4ef90f3fe,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,2,Pengiriman lama dan makanan sudah dingin.,2026-02-11,2026-02-16 22:49:52.637856,Pengiriman lama dan makanan sudah dingin.,passage: Pengiriman lama dan makanan sudah din...,1
2,92d8d828-6bd3-4700-9a2f-1156e297e468,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,3,Rasanya ok tapi porsinya kecil.,2026-02-11,2026-02-16 22:49:52.637856,Rasanya ok tapi porsinya kecil.,passage: Rasanya ok tapi porsinya kecil.,3
3,d8376516-57a3-4b85-9f5b-87c6b1d9b49d,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,1,"Packaging bocor, kuah tumpah semua.",2026-02-12,2026-02-16 22:49:52.637856,"Packaging bocor, kuah tumpah semua.","passage: Packaging bocor, kuah tumpah semua.",5
4,bb14fe7c-617f-4ccb-beb2-f0ff7fa8a1f2,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,4,"Harga sesuai, pelayanan ramah.",2026-02-12,2026-02-16 22:49:52.637856,"Harga sesuai, pelayanan ramah.","passage: Harga sesuai, pelayanan ramah.",2
5,3c35c1d3-b175-49e5-814b-bd1d49eae844,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,2,CS susah dihubungi. Balasan lama.,2026-02-13,2026-02-16 22:49:52.637856,CS susah dihubungi. Balasan lama.,passage: CS susah dihubungi. Balasan lama.,1
6,31cb85e2-59ad-4093-99a2-b78b9dce480e,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,5,Cepat banget dan rasanya konsisten.,2026-02-13,2026-02-16 22:49:52.637856,Cepat banget dan rasanya konsisten.,passage: Cepat banget dan rasanya konsisten.,0
7,0d885c41-666b-4624-934e-13f8f9f7f3b3,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,3,Tempatnya agak sempit tapi bersih.,2026-02-14,2026-02-16 22:49:52.637856,Tempatnya agak sempit tapi bersih.,passage: Tempatnya agak sempit tapi bersih.,3
8,07b4ac90-481a-4cd6-b865-adb683891406,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,4,"Promo menarik, jadi worth it.",2026-02-14,2026-02-16 22:49:52.637856,"Promo menarik, jadi worth it.","passage: Promo menarik, jadi worth it.",3
9,5c51f552-ef87-4266-867a-3196377602ad,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,2,Terlalu asin buat saya.,2026-02-15,2026-02-16 22:49:52.637856,Terlalu asin buat saya.,passage: Terlalu asin buat saya.,4


In [19]:
# Proofreading each cluster to check if groupings are already accurate
for i in range(k):
    print(f"\n===== Cluster {i} =====")
    sample_texts = df[df["cluster"] == i]["text"].head(5).tolist()
    for t in sample_texts:
        print("-", t)


===== Cluster 0 =====
- Cepat banget dan rasanya konsisten.
- Pelayanannya cepat dan ramah.
- Pengiriman cepat dan masih hangat.
- Pesanan datang terlambat hampir satu jam.
- Pelayanan cukup cepat meskipun ramai.

===== Cluster 1 =====
- Pengiriman lama dan makanan sudah dingin.
- CS susah dihubungi. Balasan lama.
- Pengiriman lama dan makanan sudah dingin.
- CS susah dihubungi dan balasan lama.
- Salah kirim menu, tidak sesuai pesanan.

===== Cluster 2 =====
- Makanannya enak banget! Bakal balik lagi.
- Harga sesuai, pelayanan ramah.
- Makanannya enak banget! Bakal balik lagi.
- Harga sesuai kualitas, recommended.
- Harga normal untuk area ini.

===== Cluster 3 =====
- Rasanya ok tapi porsinya kecil.
- Tempatnya agak sempit tapi bersih.
- Promo menarik, jadi worth it.
- Rasanya mantap dan porsinya pas.
- Rasanya ok tapi porsinya kecil.

===== Cluster 4 =====
- Terlalu asin buat saya.
- Rasanya terlalu asin buat saya.
- Terlalu lama menunggu pesanan.

===== Cluster 5 =====
- Packaging

In [20]:
# Finding review representatives for each cluster
from sklearn.metrics.pairwise import cosine_similarity

representatives = {}

# Finding reviews closest to the centroid and use it to represent the cluster
for cluster_id in range(k):
    cluster_indices = np.where(df['cluster'] == cluster_id)[0] # Finding which row is True
    cluster_embeddings = embeddings[cluster_indices] # Only select the embeddings of the rows which are true
    centroid = kmeans.cluster_centers_[cluster_id].reshape(1,-1) # Finding the centroid of the cluster and reshaping so that cosine works
    sims = cosine_similarity(cluster_embeddings, centroid).flatten()
    best_idx = cluster_indices[np.argmax(sims)]
    representatives[cluster_id] = df.iloc[best_idx]['text_clean']

representatives

{0: 'Pelayanannya cepat dan ramah.',
 1: 'Pengiriman lama dan makanan sudah dingin.',
 2: 'Harga sesuai, pelayanan ramah.',
 3: 'Rasanya ok tapi porsinya kecil.',
 4: 'Terlalu asin buat saya.',
 5: 'Packaging bocor, kuah tumpah semua.'}

In [21]:
# Finding top keywords for each cluster
from sklearn.feature_extraction.text import TfidfVectorizer

indonesian_stopwords = [
    "dan", "yang", "untuk", "dengan", "atau", "ini", "itu",
    "karena", "jadi", "tapi", "buat", "saya", "kami", "kita",
    "di", "ke", "dari", "ada", "lebih", "sudah"
]

def top_keywords(texts, n=5):
    v = TfidfVectorizer(max_features=2000, stop_words=indonesian_stopwords)
    X = v.fit_transform(texts) # Vectorize the texts
    words = v.get_feature_names_out() # Getting the vocabulary out
    scores = X.sum(axis=0).A1 # Gives a total TF-IDF score per word
    top_idx = scores.argsort()[-n:][::-1] # Sort indices from smallest to largest, take last n indices, reverse order so highest first
    return [words[i] for i in top_idx]

In [22]:
cluster_keywords = {}
cluster_sizes = {}

for cid in range(k):
    texts = df[df['cluster']==cid]['text_clean'].tolist()
    cluster_sizes[cid] = len(texts)
    cluster_keywords[cid] = top_keywords(texts, 5)

cluster_keywords

{0: ['cepat', 'pelayanannya', 'banget', 'konsisten', 'ramah'],
 1: ['lama', 'pengiriman', 'makanan', 'dingin', 'susah'],
 2: ['harga', 'ramah', 'sesuai', 'lagi', 'enak'],
 3: ['porsinya', 'rasanya', 'kecil', 'ok', 'worth'],
 4: ['terlalu', 'asin', 'rasanya', 'pesanan', 'menunggu'],
 5: ['tumpah', 'semua', 'packaging', 'kuah', 'bocor']}

In [23]:
cluster_sizes

{0: 8, 1: 7, 2: 10, 3: 10, 4: 3, 5: 2}

In [24]:
cluster_labels = {}

for cid in cluster_keywords:
    cluster_labels[cid] = ' / '.join(cluster_keywords[cid][:3])

In [25]:
df['cluster_label'] = df['cluster'].map(cluster_labels)

In [26]:
df

,review_id,business_id,source,rating,text,review_date,created_at,text_clean,input_text,cluster,cluster_label
0,e91be263-ec1d-4e92-b399-877546206b10,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,5,Makanannya enak banget! Bakal balik lagi.,2026-02-10,2026-02-16 22:49:52.637856,Makanannya enak banget! Bakal balik lagi.,passage: Makanannya enak banget! Bakal balik l...,2,harga / ramah / sesuai
1,0bed4fca-68d6-4d27-9ef5-68f4ef90f3fe,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,2,Pengiriman lama dan makanan sudah dingin.,2026-02-11,2026-02-16 22:49:52.637856,Pengiriman lama dan makanan sudah dingin.,passage: Pengiriman lama dan makanan sudah din...,1,lama / pengiriman / makanan
2,92d8d828-6bd3-4700-9a2f-1156e297e468,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,3,Rasanya ok tapi porsinya kecil.,2026-02-11,2026-02-16 22:49:52.637856,Rasanya ok tapi porsinya kecil.,passage: Rasanya ok tapi porsinya kecil.,3,porsinya / rasanya / kecil
3,d8376516-57a3-4b85-9f5b-87c6b1d9b49d,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,1,"Packaging bocor, kuah tumpah semua.",2026-02-12,2026-02-16 22:49:52.637856,"Packaging bocor, kuah tumpah semua.","passage: Packaging bocor, kuah tumpah semua.",5,tumpah / semua / packaging
4,bb14fe7c-617f-4ccb-beb2-f0ff7fa8a1f2,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,4,"Harga sesuai, pelayanan ramah.",2026-02-12,2026-02-16 22:49:52.637856,"Harga sesuai, pelayanan ramah.","passage: Harga sesuai, pelayanan ramah.",2,harga / ramah / sesuai
5,3c35c1d3-b175-49e5-814b-bd1d49eae844,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,2,CS susah dihubungi. Balasan lama.,2026-02-13,2026-02-16 22:49:52.637856,CS susah dihubungi. Balasan lama.,passage: CS susah dihubungi. Balasan lama.,1,lama / pengiriman / makanan
6,31cb85e2-59ad-4093-99a2-b78b9dce480e,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,5,Cepat banget dan rasanya konsisten.,2026-02-13,2026-02-16 22:49:52.637856,Cepat banget dan rasanya konsisten.,passage: Cepat banget dan rasanya konsisten.,0,cepat / pelayanannya / banget
7,0d885c41-666b-4624-934e-13f8f9f7f3b3,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,3,Tempatnya agak sempit tapi bersih.,2026-02-14,2026-02-16 22:49:52.637856,Tempatnya agak sempit tapi bersih.,passage: Tempatnya agak sempit tapi bersih.,3,porsinya / rasanya / kecil
8,07b4ac90-481a-4cd6-b865-adb683891406,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,4,"Promo menarik, jadi worth it.",2026-02-14,2026-02-16 22:49:52.637856,"Promo menarik, jadi worth it.","passage: Promo menarik, jadi worth it.",3,porsinya / rasanya / kecil
9,5c51f552-ef87-4266-867a-3196377602ad,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,2,Terlalu asin buat saya.,2026-02-15,2026-02-16 22:49:52.637856,Terlalu asin buat saya.,passage: Terlalu asin buat saya.,4,terlalu / asin / rasanya


### Turning clusters into insights table

In [27]:
# Manually mapping rating to sentiment without model yet
def rating_to_sentiment(r):
    if r >= 4: return 'positive'
    if r == 3: return 'neutral'
    return 'negative'

df['sentiment_rating'] = df['rating'].apply(rating_to_sentiment)
df['is_negative'] = df['sentiment'].eq('negative').astype(int)

In [28]:
# Adding negative score, impact score into dataframe
summary_labels = {}

for cid in range(k):
    subset = df[df['cluster']==cid]
    size = len(subset)
    avg_rating = subset['rating'].mean()
    neg_ratio = subset['is_negative'].mean()

    impact_score = size * neg_ratio

    summary_labels[cid] = {
        'size': size,
        'avg_rating': round(avg_rating, 2),
        'negative_ratio': round(neg_ratio, 2),
        'impact_score': round(impact_score, 2)
    }
summary_labels

{0: {'size': 8,
  'avg_rating': np.float64(4.38),
  'negative_ratio': np.float64(0.12),
  'impact_score': np.float64(1.0)},
 1: {'size': 7,
  'avg_rating': np.float64(1.71),
  'negative_ratio': np.float64(1.0),
  'impact_score': np.float64(7.0)},
 2: {'size': 10,
  'avg_rating': np.float64(4.0),
  'negative_ratio': np.float64(0.1),
  'impact_score': np.float64(1.0)},
 3: {'size': 10,
  'avg_rating': np.float64(3.5),
  'negative_ratio': np.float64(0.0),
  'impact_score': np.float64(0.0)},
 4: {'size': 3,
  'avg_rating': np.float64(2.0),
  'negative_ratio': np.float64(1.0),
  'impact_score': np.float64(3.0)},
 5: {'size': 2,
  'avg_rating': np.float64(1.0),
  'negative_ratio': np.float64(1.0),
  'impact_score': np.float64(2.0)}}

In [29]:

df['size'] = df['cluster'].map(lambda x: summary_labels[x]['size'])
df['avg_rating'] = df['cluster'].map(lambda x: summary_labels[x]['avg_rating'])
df['negative_ratio'] = df['cluster'].map(lambda x: summary_labels[x]['negative_ratio'])
df['impact_score'] = df['cluster'].map(lambda x: summary_labels[x]['impact_score'])

In [30]:
df

,review_id,business_id,source,rating,text,review_date,created_at,text_clean,input_text,cluster,cluster_label,sentiment,is_negative,size,avg_rating,negative_ratio,impact_score
0,e91be263-ec1d-4e92-b399-877546206b10,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,5,Makanannya enak banget! Bakal balik lagi.,2026-02-10,2026-02-16 22:49:52.637856,Makanannya enak banget! Bakal balik lagi.,passage: Makanannya enak banget! Bakal balik l...,2,harga / ramah / sesuai,positive,0,10,4.00,0.10,1.0
1,0bed4fca-68d6-4d27-9ef5-68f4ef90f3fe,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,2,Pengiriman lama dan makanan sudah dingin.,2026-02-11,2026-02-16 22:49:52.637856,Pengiriman lama dan makanan sudah dingin.,passage: Pengiriman lama dan makanan sudah din...,1,lama / pengiriman / makanan,negative,1,7,1.71,1.00,7.0
2,92d8d828-6bd3-4700-9a2f-1156e297e468,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,3,Rasanya ok tapi porsinya kecil.,2026-02-11,2026-02-16 22:49:52.637856,Rasanya ok tapi porsinya kecil.,passage: Rasanya ok tapi porsinya kecil.,3,porsinya / rasanya / kecil,neutral,0,10,3.50,0.00,0.0
3,d8376516-57a3-4b85-9f5b-87c6b1d9b49d,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,1,"Packaging bocor, kuah tumpah semua.",2026-02-12,2026-02-16 22:49:52.637856,"Packaging bocor, kuah tumpah semua.","passage: Packaging bocor, kuah tumpah semua.",5,tumpah / semua / packaging,negative,1,2,1.00,1.00,2.0
4,bb14fe7c-617f-4ccb-beb2-f0ff7fa8a1f2,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,4,"Harga sesuai, pelayanan ramah.",2026-02-12,2026-02-16 22:49:52.637856,"Harga sesuai, pelayanan ramah.","passage: Harga sesuai, pelayanan ramah.",2,harga / ramah / sesuai,positive,0,10,4.00,0.10,1.0
5,3c35c1d3-b175-49e5-814b-bd1d49eae844,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,2,CS susah dihubungi. Balasan lama.,2026-02-13,2026-02-16 22:49:52.637856,CS susah dihubungi. Balasan lama.,passage: CS susah dihubungi. Balasan lama.,1,lama / pengiriman / makanan,negative,1,7,1.71,1.00,7.0
6,31cb85e2-59ad-4093-99a2-b78b9dce480e,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,5,Cepat banget dan rasanya konsisten.,2026-02-13,2026-02-16 22:49:52.637856,Cepat banget dan rasanya konsisten.,passage: Cepat banget dan rasanya konsisten.,0,cepat / pelayanannya / banget,positive,0,8,4.38,0.12,1.0
7,0d885c41-666b-4624-934e-13f8f9f7f3b3,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,3,Tempatnya agak sempit tapi bersih.,2026-02-14,2026-02-16 22:49:52.637856,Tempatnya agak sempit tapi bersih.,passage: Tempatnya agak sempit tapi bersih.,3,porsinya / rasanya / kecil,neutral,0,10,3.50,0.00,0.0
8,07b4ac90-481a-4cd6-b865-adb683891406,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,4,"Promo menarik, jadi worth it.",2026-02-14,2026-02-16 22:49:52.637856,"Promo menarik, jadi worth it.","passage: Promo menarik, jadi worth it.",3,porsinya / rasanya / kecil,positive,0,10,3.50,0.00,0.0
9,5c51f552-ef87-4266-867a-3196377602ad,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,2,Terlalu asin buat saya.,2026-02-15,2026-02-16 22:49:52.637856,Terlalu asin buat saya.,passage: Terlalu asin buat saya.,4,terlalu / asin / rasanya,negative,1,3,2.00,1.00,3.0


### Add topics into postgresql (label, keywords, size)

In [35]:
import json
from sqlalchemy import text

run_id = 'b3966cda-220f-42d1-a5fb-490b6b584e0d'

query = text("""
    INSERT INTO topic (run_id, label, keywords, size)
    VALUES (:run_id, :label, :keywords, :size)
    RETURNING topic_id
""")

cluster_to_topic_id = {} # Store the topic_id of each cluster

with engine.begin() as conn: # Opens a transaction
    for cid in range(k):
        label = cluster_labels[cid]
        keywords = cluster_keywords[cid]
        size = summary_labels[cid]['size']
        topic_id = conn.execute(query, {
            'run_id': run_id,
            'label': label,
            'keywords': json.dumps(keywords), # Convert python list to valid JSON
            'size': size
        }).scalar() # Scalar returns the first column of the first row
        cluster_to_topic_id[cid] = topic_id

cluster_to_topic_id

{0: UUID('8435e993-66e8-46a6-af0b-558cbf1d9be4'),
 1: UUID('62608dbf-1e69-4186-9979-ea877144182a'),
 2: UUID('12db6ae9-aad7-450b-a9ea-e98aeda95288'),
 3: UUID('f7309b8b-b88c-4e22-a4e4-36cf6dad84fd'),
 4: UUID('47b91498-92a4-424a-be15-d7047a4521b0'),
 5: UUID('a1b50066-30f0-4eca-aa7e-f898d9937cd7')}

In [36]:
r = df.iloc[0]

review_id = r["review_id"]
cluster = int(r["cluster"])
topic_id = cluster_to_topic_id[cluster]

print(review_id, cluster, topic_id)


e91be263-ec1d-4e92-b399-877546206b10 2 12db6ae9-aad7-450b-a9ea-e98aeda95288


### Match reviews with topics in PostGreSQL through Review Analaysis

In [40]:
query_review_analysis = text("""
    INSERT INTO review_analysis (review_id, run_id, topic_id)
    VALUES (:review_id, :run_id, :topic_id)
""")

In [41]:
rows = []
for _, r in df.iterrows():
    rows.append({
        'review_id': r['review_id'],
        'run_id': run_id,
        'topic_id': cluster_to_topic_id[r['cluster']]
    })

with engine.begin() as conn:
    conn.execute(query_review_analysis, rows)

### Sentiment Analysis

In [43]:
from transformers import pipeline

sentiment_model = pipeline(
    'text-classification',
    model = 'w11wo/indonesian-roberta-base-sentiment-classifier',
    device=0
)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: w11wo/indonesian-roberta-base-sentiment-classifier
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/328 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [45]:
sentiment_model("Pengiriman lama dan makanan sudah dingin")


[{'label': 'negative', 'score': 0.9949117302894592}]

In [50]:
texts = df['text_clean'].tolist()

results = sentiment_model(texts, batch_size=16)

df['sentiment_label_model'] = [r['label'].lower() for r in results]
df['sentiment_score_model'] = [r['score'] for r in results]
df[['text_clean', 'rating', 'sentiment_rating', 'sentiment_label_model', 'sentiment_score_model']]

,text_clean,rating,sentiment_rating,sentiment_label_model,sentiment_score_model
0,Makanannya enak banget! Bakal balik lagi.,5,positive,positive,0.998236
1,Pengiriman lama dan makanan sudah dingin.,2,negative,negative,0.994191
2,Rasanya ok tapi porsinya kecil.,3,neutral,negative,0.683876
3,"Packaging bocor, kuah tumpah semua.",1,negative,negative,0.995752
4,"Harga sesuai, pelayanan ramah.",4,positive,positive,0.954170
5,CS susah dihubungi. Balasan lama.,2,negative,negative,0.998266
6,Cepat banget dan rasanya konsisten.,5,positive,positive,0.982166
7,Tempatnya agak sempit tapi bersih.,3,neutral,positive,0.990528
8,"Promo menarik, jadi worth it.",4,positive,positive,0.980327
9,Terlalu asin buat saya.,2,negative,negative,0.992302


In [ ]:
# Update sentiments to postgre table review_analysis

update_review_analysis_query = text(
    """
    UPDATE review_analysis
    SET sentiment_label = :sentiment_label,
    sentiment_score = :sentiment_score
    WHERE review_id = :review_id
    AND run_id = :run_id
    """
)

payload = []

for _, r in df.iterrows():
    payload.append({
        "sentiment_label": r['sentiment_label_model'],
        'sentiment_score': r['sentiment_score_model'],
        'review_id': r['review_id'],
        'run_id': run_id
    })

with engine.begin() as conn:
    conn.execute(update_review_analysis_query, payload)

In [51]:
df

,review_id,business_id,source,rating,text,review_date,created_at,text_clean,input_text,cluster,cluster_label,sentiment_rating,is_negative,size,avg_rating,negative_ratio,impact_score,sentiment_label_model,sentiment_score_model
0,e91be263-ec1d-4e92-b399-877546206b10,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,5,Makanannya enak banget! Bakal balik lagi.,2026-02-10,2026-02-16 22:49:52.637856,Makanannya enak banget! Bakal balik lagi.,passage: Makanannya enak banget! Bakal balik l...,2,harga / ramah / sesuai,positive,0,10,4.00,0.10,1.0,positive,0.998236
1,0bed4fca-68d6-4d27-9ef5-68f4ef90f3fe,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,2,Pengiriman lama dan makanan sudah dingin.,2026-02-11,2026-02-16 22:49:52.637856,Pengiriman lama dan makanan sudah dingin.,passage: Pengiriman lama dan makanan sudah din...,1,lama / pengiriman / makanan,negative,1,7,1.71,1.00,7.0,negative,0.994191
2,92d8d828-6bd3-4700-9a2f-1156e297e468,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,3,Rasanya ok tapi porsinya kecil.,2026-02-11,2026-02-16 22:49:52.637856,Rasanya ok tapi porsinya kecil.,passage: Rasanya ok tapi porsinya kecil.,3,porsinya / rasanya / kecil,neutral,0,10,3.50,0.00,0.0,negative,0.683876
3,d8376516-57a3-4b85-9f5b-87c6b1d9b49d,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,1,"Packaging bocor, kuah tumpah semua.",2026-02-12,2026-02-16 22:49:52.637856,"Packaging bocor, kuah tumpah semua.","passage: Packaging bocor, kuah tumpah semua.",5,tumpah / semua / packaging,negative,1,2,1.00,1.00,2.0,negative,0.995752
4,bb14fe7c-617f-4ccb-beb2-f0ff7fa8a1f2,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,4,"Harga sesuai, pelayanan ramah.",2026-02-12,2026-02-16 22:49:52.637856,"Harga sesuai, pelayanan ramah.","passage: Harga sesuai, pelayanan ramah.",2,harga / ramah / sesuai,positive,0,10,4.00,0.10,1.0,positive,0.954170
5,3c35c1d3-b175-49e5-814b-bd1d49eae844,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,2,CS susah dihubungi. Balasan lama.,2026-02-13,2026-02-16 22:49:52.637856,CS susah dihubungi. Balasan lama.,passage: CS susah dihubungi. Balasan lama.,1,lama / pengiriman / makanan,negative,1,7,1.71,1.00,7.0,negative,0.998266
6,31cb85e2-59ad-4093-99a2-b78b9dce480e,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,5,Cepat banget dan rasanya konsisten.,2026-02-13,2026-02-16 22:49:52.637856,Cepat banget dan rasanya konsisten.,passage: Cepat banget dan rasanya konsisten.,0,cepat / pelayanannya / banget,positive,0,8,4.38,0.12,1.0,positive,0.982166
7,0d885c41-666b-4624-934e-13f8f9f7f3b3,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,3,Tempatnya agak sempit tapi bersih.,2026-02-14,2026-02-16 22:49:52.637856,Tempatnya agak sempit tapi bersih.,passage: Tempatnya agak sempit tapi bersih.,3,porsinya / rasanya / kecil,neutral,0,10,3.50,0.00,0.0,positive,0.990528
8,07b4ac90-481a-4cd6-b865-adb683891406,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,4,"Promo menarik, jadi worth it.",2026-02-14,2026-02-16 22:49:52.637856,"Promo menarik, jadi worth it.","passage: Promo menarik, jadi worth it.",3,porsinya / rasanya / kecil,positive,0,10,3.50,0.00,0.0,positive,0.980327
9,5c51f552-ef87-4266-867a-3196377602ad,65bdf616-863b-4b42-9817-f7f18662d45e,google_maps,2,Terlalu asin buat saya.,2026-02-15,2026-02-16 22:49:52.637856,Terlalu asin buat saya.,passage: Terlalu asin buat saya.,4,terlalu / asin / rasanya,negative,1,3,2.00,1.00,3.0,negative,0.992302
